# Part 1 — Does DoRA's learning pattern transfer to a vision-language model?

**Paper claim (DoRA, Liu et al. 2024, Fig. 2):** decompose each weight into magnitude and direction. Across layers and
checkpoints, full fine-tuning shows a **negative** correlation between magnitude change ΔM and direction change ΔD; LoRA
shows a **positive** one; DoRA's pattern resembles full fine-tuning (negative).

**Hypothesis:** the same holds when SmolVLM-256M is fine-tuned on DocVQA — in the language model *and* in the vision
encoder (bonus: per-tower analysis: vision encoder / connector / LM).

Setup: SmolVLM-256M-Instruct · DocVQA subset (2000 train / 300 validation) · LoRA vs DoRA (r=8, α=16) vs full FT on the
same targets (q,k,v,o of both towers + connector) · 1 epoch · ΔM/ΔD recorded at 8 checkpoints · ANLS for accuracy.
Runtime: T4 GPU. Each training run ≈ 15–30 min; every result is saved to Drive, so cells can be re-run after a disconnect.

In [ ]:
# ---- config: code comes from GitHub, results go to Google Drive ----
REPO_URL = 'https://github.com/rivindu02/parameter-efficient-finetuning.git'   # <- your repo
BRANCH = 'main'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.46" peft accelerate datasets
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/EN4554/part1_vlm_dora'
os.makedirs(OUT, exist_ok=True)
url = REPO_URL
try:  # private repo: add a Colab secret GH_TOKEN (read-only fine-grained token) with notebook access
    from google.colab import userdata
    url = REPO_URL.replace('https://', f"https://{userdata.get('GH_TOKEN')}@")
except Exception:
    pass
if not os.path.exists('/content/repo/.git'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/repo'], check=True)
else:
    subprocess.run(['git', '-C', '/content/repo', 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', '/content/repo', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
%cd /content/repo/part1_vlm_dora
sys.path.insert(0, '/content/repo/part1_vlm_dora')
SEED = 42
print('outputs ->', OUT)

MODULES = ['dora_vlm']
!pip -q install matplotlib

In [ ]:
# re-run after pushing code changes: pulls the latest commit and reloads modules
import importlib, subprocess
print(subprocess.run(['git', '-C', '/content/repo', 'pull'], capture_output=True, text=True).stdout)
for m in list(sys.modules):
    if m in MODULES: importlib.reload(sys.modules[m])

## 1. Config + data subset (cached as JPEG on local disk)

In [ ]:
import json, os, torch, gc
import dora_vlm as dv
SMOKE = False   # True: tiny run (16 train / 8 eval, ~5 min) to check the pipeline end to end
if SMOKE:
    cfg = dv.Config(out_dir=OUT + '/smoke', data_dir='/content/data_docvqa_smoke', seed=SEED,
                    n_train=16, n_eval=8, n_snapshots=2, eval_batch=4)
else:
    cfg = dv.Config(out_dir=OUT, data_dir='/content/data_docvqa', seed=SEED)
print(cfg)
dv.prepare_data(cfg)
dv.show_masking(cfg)   # only the answer tokens should be supervised

## 2. Zero-shot baseline

In [ ]:
if not os.path.exists(os.path.join(OUT, 'eval_zeroshot.json')):
    model, proc = dv.load_model_and_processor(cfg)
    dv.evaluate_anls(cfg, model.to('cuda'), proc, 'zeroshot')
    del model; gc.collect(); torch.cuda.empty_cache()

## 3. Training runs (one cell each, skipped if results already on Drive)

In [ ]:
if not os.path.exists(os.path.join(OUT, 'eval_lora.json')):
    model, proc = dv.train_method(cfg, 'lora')
    dv.evaluate_anls(cfg, model, proc, 'lora')
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
if not os.path.exists(os.path.join(OUT, 'eval_dora.json')):
    model, proc = dv.train_method(cfg, 'dora')
    dv.evaluate_anls(cfg, model, proc, 'dora')
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
if not os.path.exists(os.path.join(OUT, 'eval_full.json')):
    model, proc = dv.train_method(cfg, 'full')
    dv.evaluate_anls(cfg, model, proc, 'full')
    del model; gc.collect(); torch.cuda.empty_cache()

## 4. Accuracy: zero-shot vs adapted

In [ ]:
rows = []
for n in ['zeroshot', 'lora', 'dora', 'full']:
    p = os.path.join(OUT, f'eval_{n}.json')
    if os.path.exists(p):
        r = json.load(open(p)); rows.append((n, r['anls'], r['exact_match']))
print(f"{'model':<10}{'ANLS':>8}{'EM':>8}")
for n, a, e in rows: print(f'{n:<10}{a:>8.2f}{e:>8.2f}')

## 5. Claim test: ΔM vs ΔD correlation
`r_pooled`: all (layer, checkpoint) points, as in the paper's Fig. 2. `r_within`: mean across-layer correlation per checkpoint
(removes the trivial "both grow over time" trend). Paper's prediction: **full < 0, LoRA > 0, DoRA < 0**.
First q/v projections (the modules analysed in the paper), then all projections.

In [ ]:
corr_qv = dv.analyze(OUT, proj_filter=('q_proj', 'v_proj'))

In [ ]:
corr_all = dv.analyze(OUT, proj_filter=None)
json.dump({'qv': corr_qv, 'all': corr_all}, open(os.path.join(OUT, 'correlations.json'), 'w'), indent=1)

## 6. Fig. 2-style scatter plots, per tower

In [ ]:
from IPython.display import Image, display
for tower in ['text', 'vision']:
    for proj in ['q_proj', 'v_proj']:
        display(Image(dv.plot_scatter(OUT, tower=tower, proj=proj)))

## 7. Bonus: where does the adaptation happen?
Mean ΔD and ΔM per tower at the last checkpoint: does each method move the vision encoder, the connector or the language
model the most, and does DoRA's magnitude/direction split behave differently in the vision encoder?

In [ ]:
from collections import defaultdict
for m in ['full', 'lora', 'dora']:
    p = os.path.join(OUT, f'stats_{m}.json')
    if not os.path.exists(p): continue
    st = json.load(open(p))['stats']; last = max(r['step'] for r in st)
    agg = defaultdict(list)
    for r in st:
        if r['step'] == last: agg[r['tower']].append(r)
    print(m, {t: dict(dD=f"{sum(r['dD'] for r in v)/len(v):.2e}", dM=f"{sum(r['dM'] for r in v)/len(v):.2e}", n=len(v))
              for t, v in agg.items()})